In [ ]:
import pandas as pd
import re
from datetime import datetime
import json

class DataValidator:
    def __init__(self):
        self.validation_rules = {
            'date_formats': [
                r'\d{2}/\d{2}/\d{4}',  # DD/MM/YYYY
                r'\d{1,2}/\d{1,2}/\d{4}',  # D/M/YYYY
            ],
            'sales_order_pattern': r'1-\d{6}',
            'valid_names': ['KEN', 'TOM', 'JOHN', 'MICHAEL', 'CLAYTON', 'ZOEWE'],
            'valid_showrooms': ['Nova TradeHub', 'NV-TH'],
            'valid_reasons': ['Walk-in', 'Facebook', 'Repeat', 'Appointment', 'Referral', 'Sales Up'],
            'amount_range': (0, 50000)  # Reasonable amount range
        }
        
        self.day_mapping = {
            'monday': 1, 'tuesday': 2, 'wednesday': 3,
            'thursday': 4, 'friday': 5, 'saturday': 6, 'sunday': 7
        }
    
    def validate_row(self, row):
        """Validate a single data row"""
        errors = []
        warnings = []
        
        # Validate Date
        if pd.isna(row.get('Date')) or not row.get('Date'):
            errors.append("Missing date")
        else:
            date_valid = False
            for pattern in self.validation_rules['date_formats']:
                if re.match(pattern, str(row['Date'])):
                    date_valid = True
                    break
            if not date_valid:
                errors.append(f"Invalid date format: {row['Date']}")
        
        # Validate Day of the Week
        if pd.isna(row.get('Day of the Week')):
            errors.append("Missing day of the week")
        
        # Validate Day Number
        day_num = row.get('Number corresponding to the day')
        if pd.isna(day_num) or day_num not in range(1, 8):
            errors.append(f"Invalid day number: {day_num}")
        
        # Validate Name
        if pd.isna(row.get('Name')) or row.get('Name') not in self.validation_rules['valid_names']:
            errors.append(f"Invalid or missing name: {row.get('Name')}")
        
        # Validate Showroom
        if pd.isna(row.get('Showroom')) or row.get('Showroom') not in self.validation_rules['valid_showrooms']:
            warnings.append(f"Unusual showroom: {row.get('Showroom')}")
        
        # Validate Sales Order Number
        sales_order = row.get('Sales Order No.')
        if pd.isna(sales_order) or not re.match(self.validation_rules['sales_order_pattern'], str(sales_order)):
            errors.append(f"Invalid sales order number: {sales_order}")
        
        # Validate Amount
        amount = row.get('Amount')
        if pd.isna(amount):
            errors.append("Missing amount")
        else:
            try:
                amount_float = float(amount)
                min_amt, max_amt = self.validation_rules['amount_range']
                if not (min_amt <= amount_float <= max_amt):
                    warnings.append(f"Amount outside expected range: {amount_float}")
            except (ValueError, TypeError):
                errors.append(f"Invalid amount format: {amount}")
        
        # Validate Reason
        reason = row.get('Reason')
        if pd.isna(reason) or reason not in self.validation_rules['valid_reasons']:
            warnings.append(f"Unusual reason: {reason}")
        
        return {
            'valid': len(errors) == 0,
            'errors': errors,
            'warnings': warnings
        }
    
    def validate_dataset(self, df):
        """Validate entire dataset"""
        validation_results = {
            'total_rows': len(df),
            'valid_rows': 0,
            'rows_with_errors': 0,
            'rows_with_warnings': 0,
            'detailed_results': [],
            'summary': {
                'missing_dates': 0,
                'invalid_sales_orders': 0,
                'missing_names': 0,
                'invalid_amounts': 0,
                'duplicate_sales_orders': 0
            }
        }
        
        # Check for duplicate sales orders
        duplicate_orders = df['Sales Order No.'].duplicated().sum()
        validation_results['summary']['duplicate_sales_orders'] = duplicate_orders
        
        for index, row in df.iterrows():
            row_validation = self.validate_row(row)
            row_validation['row_index'] = index
            validation_results['detailed_results'].append(row_validation)
            
            if row_validation['valid']:
                validation_results['valid_rows'] += 1
            else:
                validation_results['rows_with_errors'] += 1
            
            if row_validation['warnings']:
                validation_results['rows_with_warnings'] += 1
            
            # Update summary counts
            for error in row_validation['errors']:
                if 'date' in error.lower():
                    validation_results['summary']['missing_dates'] += 1
                elif 'sales order' in error.lower():
                    validation_results['summary']['invalid_sales_orders'] += 1
                elif 'name' in error.lower():
                    validation_results['summary']['missing_names'] += 1
                elif 'amount' in error.lower():
                    validation_results['summary']['invalid_amounts'] += 1
        
        return validation_results
    
    def generate_validation_report(self, validation_results, output_file='validation_report.txt'):
        """Generate a detailed validation report"""
        with open(output_file, 'w') as f:
            f.write("DATA VALIDATION REPORT\n")
            f.write("=" * 50 + "\n\n")
            
            # Summary
            f.write("SUMMARY:\n")
            f.write(f"Total rows processed: {validation_results['total_rows']}\n")
            f.write(f"Valid rows: {validation_results['valid_rows']}\n")
            f.write(f"Rows with errors: {validation_results['rows_with_errors']}\n")
            f.write(f"Rows with warnings: {validation_results['rows_with_warnings']}\n")
            f.write(f"Accuracy: {validation_results['valid_rows']/validation_results['total_rows']*100:.1f}%\n\n")
            
            # Detailed issues
            f.write("DETAILED ISSUES:\n")
            f.write("-" * 30 + "\n")
            for i, result in enumerate(validation_results['detailed_results']):
                if result['errors'] or result['warnings']:
                    f.write(f"Row {result['row_index'] + 1}:\n")
                    for error in result['errors']:
                        f.write(f"  ERROR: {error}\n")
                    for warning in result['warnings']:
                        f.write(f"  WARNING: {warning}\n")
                    f.write("\n")
        
        print(f"Validation report saved to {output_file}")
    
    def fix_common_issues(self, df):
        """Attempt to fix common OCR errors"""
        df_fixed = df.copy()
        
        # Fix common name OCR errors
        name_fixes = {
            'KFN': 'KEN',
            'T0M': 'TOM',
            'J0HN': 'JOHN',
            'CLAYTO': 'CLAYTON',
            'CLAYT0N': 'CLAYTON'
        }
        
        for col in ['Name']:
            if col in df_fixed.columns:
                df_fixed[col] = df_fixed[col].replace(name_fixes)
        
        # Fix sales order format
        if 'Sales Order No.' in df_fixed.columns:
            df_fixed['Sales Order No.'] = df_fixed['Sales Order No.'].astype(str).str.replace('O', '0')
        
        # Fix amount formatting
        if 'Amount' in df_fixed.columns:
            df_fixed['Amount'] = pd.to_numeric(df_fixed['Amount'], errors='coerce')
        
        return df_fixed

class ManualReviewInterface:
    """Simple interface for manual review of flagged items"""
    
    def __init__(self, df, validation_results):
        self.df = df
        self.validation_results = validation_results
    
    def review_errors(self):
        """Interactive review of rows with errors"""
        print("MANUAL REVIEW INTERFACE")
        print("=" * 40)
        
        error_rows = [r for r in self.validation_results['detailed_results'] if r['errors']]
        
        for i, row_result in enumerate(error_rows):
            row_idx = row_result['row_index']
            row_data = self.df.iloc[row_idx]
            
            print(f"\nRow {row_idx + 1} of {len(self.df)} (Error {i+1} of {len(error_rows)}):")
            print("-" * 30)
            
            # Display row data
            for col, value in row_data.items():
                print(f"{col}: {value}")
            
            print("\nErrors found:")
            for error in row_result['errors']:
                print(f"  - {error}")
            
            print("\nOptions:")
            print("1. Edit this row")
            print("2. Delete this row") 
            print("3. Skip (keep as-is)")
            print("4. Finish review")
            
            choice = input("Enter choice (1-4): ").strip()
            
            if choice == '1':
                self.edit_row(row_idx)
            elif choice == '2':
                self.df = self.df.drop(row_idx).reset_index(drop=True)
                print("Row deleted.")
            elif choice == '3':
                continue
            elif choice == '4':
                break
        
        return self.df
    
    def edit_row(self, row_idx):
        """Edit a specific row"""
        print(f"\nEditing row {row_idx + 1}:")
        row = self.df.iloc[row_idx].copy()
        
        for col in self.df.columns:
            current_value = row[col]
            new_value = input(f"{col} (current: {current_value}): ").strip()
            
            if new_value:
                if col == 'Amount':
                    try:
                        new_value = float(new_value)
                    except ValueError:
                        print("Invalid amount, keeping original")
                        continue
                elif col == 'Number corresponding to the day':
                    try:
                        new_value = int(new_value)
                    except ValueError:
                        print("Invalid day number, keeping original")
                        continue
                
                self.df.iloc[row_idx, self.df.columns.get_loc(col)] = new_value
        
        print("Row updated!")

# Usage example
if __name__ == "__main__":
    # Load your extracted data
    df = pd.read_csv('extracted_sales_data.csv')
    
    # Validate
    validator = DataValidator()
    results = validator.validate_dataset(df)
    
    # Generate report
    validator.generate_validation_report(results)
    
    # Try automatic fixes
    df_fixed = validator.fix_common_issues(df)
    
    # Manual review if needed
    if results['rows_with_errors'] > 0:
        print(f"\nFound {results['rows_with_errors']} rows with errors.")
        review = input("Start manual review? (y/n): ").lower().strip()
        
        if review == 'y':
            reviewer = ManualReviewInterface(df_fixed, results)
            df_final = reviewer.review_errors()
            df_final.to_csv('final_validated_data.csv', index=False)
            print("Final validated data saved!")